## SWAT+ prediction

## FMI POINT 2023 till today

## Cell 1: FMI Observations Downloader (2023 to Today)
## Downloads historical data and stores it in a memory variable df_obs.

In [ ]:
# ==========================================
# CELL 1: SMART STATE DETECTION & OBS DOWNLOAD 
# ==========================================
import os, datetime, time, glob, shutil, pandas as pd, numpy as np
from fmiopendata.wfs import download_stored_query

# --- 1. Path Configuration ---
base_dir = "/home/jovyan/Taki_Thesis/summery"
original_txtinout = os.path.join(base_dir, "TxtInOut")
obs_archives = sorted(glob.glob(os.path.join(base_dir, "TxtInOut_Obs_Only_*")))
source_obs_folder = obs_archives[-1] if obs_archives else original_txtinout

def get_last_date(folder):
    pcp_files = glob.glob(os.path.join(folder, "*.pcp"))
    if not pcp_files: return None
    with open(pcp_files[0], 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
        if len(lines) < 4: return None
        p = lines[-1].split()
        return datetime.datetime(int(p[0]), 1, 1) + datetime.timedelta(int(p[1]) - 1)

last_obs_date = get_last_date(source_obs_folder)
fetch_start = last_obs_date + datetime.timedelta(days=1)
fetch_end = (datetime.datetime.now() - datetime.timedelta(days=1))

print(f"Source: {os.path.basename(source_obs_folder)} | Last Date: {last_obs_date.date()}")

# --- 2. Incremental Download ---
obs_records = []
if fetch_start <= fetch_end:
    start_time = time.time()
    curr = fetch_start
    chunks = []
    while curr <= fetch_end:
        chunks.append((curr, min(curr + datetime.timedelta(days=6), fetch_end)))
        curr = chunks[-1][1] + datetime.timedelta(days=1)

    for i, (s, e) in enumerate(chunks):
        s_str, e_str = s.strftime("%Y-%m-%dT00:00:00Z"), e.strftime("%Y-%m-%dT23:59:59Z")
        try:
            o_daily = download_stored_query("fmi::observations::weather::daily::multipointcoverage", args=["fmisid=101887", "starttime="+s_str, "endtime="+e_str])
            o_hourly = download_stored_query("fmi::observations::weather::multipointcoverage", args=["fmisid=101887", "starttime="+s_str, "endtime="+e_str])
            o_rad = download_stored_query("fmi::observations::radiation::multipointcoverage", args=["fmisid=101932", "starttime="+s_str, "endtime="+e_str])
            
            for t in o_daily.data.keys():
                dt = pd.to_datetime(t).tz_localize(None)
                st_key = list(o_daily.data[t].keys())[0]
                row = o_daily.data[t][st_key]
                
                # FIX: Added .get('value') to extract raw numbers
                pcp_val = row.get('Precipitation amount', {}).get('value')
                tmax_val = row.get('Maximum temperature', {}).get('value')
                tmin_val = row.get('Minimum temperature', {}).get('value')

                # Hourly aggregations (already had .get('value') logic)
                h_v = [float(dh[list(dh.keys())[0]].get('Relative humidity', {}).get('value')) for th, dh in o_hourly.data.items() if dt <= pd.to_datetime(th).tz_localize(None) <= (dt + datetime.timedelta(hours=23))]
                r_v = [float(ds[list(ds.keys())[0]].get('Global radiation', {}).get('value')) for ts, ds in o_rad.data.items() if dt <= pd.to_datetime(ts).tz_localize(None) <= (dt + datetime.timedelta(hours=23))]
                
                obs_records.append({
                    'time': dt, 
                    'pcp': pcp_val, 
                    'tmax': tmax_val, 
                    'tmin': tmin_val, 
                    'hmd': (np.mean(h_v)/100.0) if h_v else np.nan, 
                    'slr': (np.sum(r_v)*0.00006) if r_v else np.nan
                })
        except Exception as err:
            print(f"\nWarning: Error in chunk {s.date()}: {err}")
        
        print(f"Progress: {((i+1)/len(chunks)*100):.1f}% | Time: {int(time.time()-start_time)}s", end="\r")

df_obs = pd.DataFrame(obs_records).set_index('time').sort_index() if obs_records else pd.DataFrame()
print("\n New observations downloaded.")

## Cell 2: Process & Append Observations
## Copies the TxtInOut folder and appends df_obs to the station files.

In [ ]:
# ==========================================
# CELL 2: CREATE DUAL FOLDERS (A & B) - FINAL
# ==========================================
import os, shutil, datetime, pandas as pd
import glob

# --- 1. Path Configuration ---
base_dir = "/home/jovyan/Taki_Thesis/summery"
today_str = datetime.datetime.now().strftime('%Y-%m-%d')
folder_a = os.path.join(base_dir, f"TxtInOut_Obs_Only_{today_str}")
folder_b = os.path.join(base_dir, f"TxtInOut_Full_Execution_{today_str}")

# Find reliable source folder from disk
obs_archives = sorted(glob.glob(os.path.join(base_dir, "TxtInOut_Obs_Only_*")))
valid_archives = [f for f in obs_archives if f != folder_a and os.path.exists(f)]
source_obs_folder = valid_archives[-1] if valid_archives else os.path.join(base_dir, "TxtInOut")

# --- 2. Manage Folders (Idempotent & Safe) ---
if not os.path.exists(folder_a):
    shutil.copytree(source_obs_folder, folder_a)
    print(f"Created Archive: {os.path.basename(folder_a)}")
else:
    print(f"Archive already exists: {os.path.basename(folder_a)}")

if os.path.exists(folder_b): shutil.rmtree(folder_b)
shutil.copytree(folder_a, folder_b)
print(f"Created Execution Folder: {os.path.basename(folder_b)}")

# --- 3. Fortran Formatting Helpers (Strict Alignment) ---
def clean_pcp(v):
    # FMI Trace (-1.0) and other negatives/NaNs become 0.0
    if isinstance(v, dict): v = v.get('value', 0.0)
    try:
        val = float(v)
        return 0.0 if (pd.isna(val) or val < 0) else val
    except:
        return 0.0

def clean_val(v):
    # General cleaner for non-precipitation files
    if isinstance(v, dict): v = v.get('value', 0.0)
    try:
        return float(v) if not pd.isna(v) else 0.0
    except:
        return 0.0

def format_swat_1val(y, j, v, is_pcp=False): 
    # Year(4) + Space(2) + Day(3) + Value(11 chars with 5 decimals)
    # The 11-char fixed width ensures the decimal points strictly align vertically
    val = clean_pcp(v) if is_pcp else clean_val(v)
    return f"{y:4d}  {j:3d}{val:11.5f}\n"

def format_swat_2val(y, j, v1, v2): 
    # Tmax and Tmin with the same 11-char fixed width logic
    return f"{y:4d}  {j:3d}{clean_val(v1):11.5f}{clean_val(v2):11.5f}\n"

# --- 4. Append Data ---
if 'df_obs' in globals() and not df_obs.empty:
    with open(os.path.join(folder_a, "weather-sta.cli"), 'r') as f:
        stations = [line.split() for line in f.readlines()[2:] if len(line.split()) >= 6]

    for target in [folder_a, folder_b]:
        print(f"Appending data to: {os.path.basename(target)}")
        for st in stations:
            # 1-Value Files: PCP, SLR, HMD
            for key, fname in [('pcp', st[2]), ('slr', st[4]), ('hmd', st[5])]:
                if fname == 'null': continue
                with open(os.path.join(target, fname), 'a') as f:
                    for dt, row in df_obs.iterrows():
                        is_pcp_file = (key == 'pcp')
                        line = format_swat_1val(dt.year, dt.timetuple().tm_yday, row[key], is_pcp_file)
                        f.write(line)
            
            # 2-Value File: Temperature
            t_fname = st[3]
            with open(os.path.join(target, t_fname), 'a') as f:
                for dt, row in df_obs.iterrows():
                    line = format_swat_2val(dt.year, dt.timetuple().tm_yday, row['tmax'], row['tmin'])
                    f.write(line)
    print("✅ Success: Spacing perfectly matches Fortran format and Trace values are fixed!")
else:
    print("✅ Folders synced. No new data in memory to append.")

## HARMOINE prediction data

## Cell 3: Harmonie Forecast Downloader
## Fetches the future 2.5 days of forecast data and stores it in df_forecast.

In [ ]:
# ==========================================
# CELL 3: HARMONIE FORECAST DOWNLOAD
# ==========================================
import requests
import xml.etree.ElementTree as ET
import pandas as pd 

print("Fetching 3-Day Harmonie Forecast...")

f_params = {
    "service": "WFS", 
    "version": "2.0.0", 
    "request": "getFeature", 
    "storedquery_id": "fmi::forecast::harmonie::surface::point::simple", 
    "latlon": "66.364,29.316", 
    "parameters": "Temperature,Humidity,Precipitation1h,RadiationGlobal"
}

f_res = requests.get("http://opendata.fmi.fi/wfs", params=f_params)
root = ET.fromstring(f_res.content)
f_list = []

for el in root.findall('.//BsWfs:BsWfsElement', {'BsWfs': 'http://xml.fmi.fi/schema/wfs/2.0'}):
    f_list.append({
        'Time': pd.to_datetime(el.find('{http://xml.fmi.fi/schema/wfs/2.0}Time').text), 
        'Param': el.find('{http://xml.fmi.fi/schema/wfs/2.0}ParameterName').text, 
        'Val': float(el.find('{http://xml.fmi.fi/schema/wfs/2.0}ParameterValue').text)
    })

f_df = pd.DataFrame(f_list).pivot_table(index='Time', columns='Param', values='Val')
df_forecast = pd.DataFrame()
df_forecast['tmax'] = f_df['Temperature'].resample('1D').max()
df_forecast['tmin'] = f_df['Temperature'].resample('1D').min()
df_forecast['pcp'] = f_df['Precipitation1h'].resample('1D').sum()
df_forecast['hmd'] = f_df['Humidity'].resample('1D').mean() / 100.0
df_forecast['slr'] = f_df['RadiationGlobal'].resample('1D').mean() * 0.0864
df_forecast.index = df_forecast.index.tz_localize(None)

# Keeping only future forecasts
df_forecast = df_forecast[df_forecast.index >= pd.Timestamp.now().normalize()]

print(f"✅ Forecast downloaded successfully for {len(df_forecast)} days. Ready to inject.")

## Cell 4: Append Harmonie Forecast & Update CLI
## Appends the forecast data and updates the nbyr in the CLI header.

In [ ]:
# ==========================================
# CELL 4: FINALIZE FOLDER B (APPEND FORECAST)
# ==========================================
import os, datetime, pandas as pd

# --- 1. Re-declare Paths (Safety check) ---
base_dir = "/home/jovyan/Taki_Thesis/summery"
today_str = datetime.datetime.now().strftime('%Y-%m-%d')
folder_a = os.path.join(base_dir, f"TxtInOut_Obs_Only_{today_str}")
folder_b = os.path.join(base_dir, f"TxtInOut_Full_Execution_{today_str}")

print(f"Finalizing execution folder: {os.path.basename(folder_b)}")

# --- 2. Fortran Formatting Helpers (Re-declared for safety) ---
def clean_pcp(v):
    if isinstance(v, dict): v = v.get('value', 0.0)
    try:
        val = float(v)
        return 0.0 if (pd.isna(val) or val < 0) else val
    except: return 0.0

def clean_val(v):
    if isinstance(v, dict): v = v.get('value', 0.0)
    try: return float(v) if not pd.isna(v) else 0.0
    except: return 0.0

def format_swat_1val(y, j, v, is_pcp=False):
    val = clean_pcp(v) if is_pcp else clean_val(v)
    return f"{y:4d}  {j:3d}{val:11.5f}\n"

def format_swat_2val(y, j, v1, v2):
    return f"{y:4d}  {j:3d}{clean_val(v1):11.5f}{clean_val(v2):11.5f}\n"

# --- 3. Read Stations ---
with open(os.path.join(folder_b, "weather-sta.cli"), 'r') as f:
    stations = [line.split() for line in f.readlines()[2:] if len(line.split()) >= 6]

# --- 4. Append Forecast Data to Folder B ONLY ---
if 'df_forecast' in globals() and not df_forecast.empty:
    for st in stations:
        # 1-Value Files: PCP, SLR, HMD
        for key, fname in [('pcp', st[2]), ('slr', st[4]), ('hmd', st[5])]:
            if fname == 'null': continue
            with open(os.path.join(folder_b, fname), 'a') as f:
                for dt, row in df_forecast.iterrows():
                    is_pcp_file = (key == 'pcp')
                    f.write(format_swat_1val(dt.year, dt.timetuple().tm_yday, row[key], is_pcp_file))
        
        # 2-Value File: Temperature
        with open(os.path.join(folder_b, st[3]), 'a') as f:
            for dt, row in df_forecast.iterrows():
                f.write(format_swat_2val(dt.year, dt.timetuple().tm_yday, row['tmax'], row['tmin']))

    # --- 5. Update nbyr (Years Header) for both folders ---
    current_year = datetime.datetime.now().year
    for target in [folder_a, folder_b]:
        cli_path = os.path.join(target, "weather-sta.cli")
        with open(cli_path, 'r') as f: lines = f.readlines()
        lines[1] = f"{(current_year - 1990 + 1):8d}    nbyr: number of years\n"
        with open(cli_path, 'w') as f: f.writelines(lines)

    print(f"\n🚀 ALL DONE! FORECAST INJECTED.")
    print(f"✅ Safe Archive (No Forecast): {os.path.basename(folder_a)}")
    print(f"✅ Ready for SWAT+ (With Forecast): {os.path.basename(folder_b)}")
else:
    print("⚠️ Error: df_forecast is empty. Please run Cell 3 again.")

## Now run the model

## pyswatplus simulation with timer

In [ ]:
# ==========================================
# CELL 5: pySWATPlus AUTOMATED RUN
# ==========================================
import pySWATPlus
import numpy
import pandas
import os
import datetime
import time  # Added to track execution time

# --- SWAT+ Simulation Setup (Automated Directories) ---
base_dir = "/home/jovyan/Taki_Thesis/summery"
today_str = datetime.datetime.now().strftime('%Y-%m-%d')
time_str = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

# Dynamically initialize the original TxtInOut directory path (from Cell 4)
txtinout_dir = os.path.join(base_dir, f"TxtInOut_Full_Execution_{today_str}")

# Initialize TxtinoutReader class for the source directory
txtinout_reader = pySWATPlus.TxtinoutReader(tio_dir=txtinout_dir)

# --- Simulation in a Custom Directory (Automated timestamp) ---
sim_dir = os.path.join(base_dir, f"prediction_model_run_pyswatplus_{time_str}")

if not os.path.exists(sim_dir):
    os.makedirs(sim_dir)

# Copy required input files and executable to the custom directory
txtinout_reader.copy_required_files(sim_dir=sim_dir)

# Initialize TxtinoutReader for the custom directory setup
sim_reader = pySWATPlus.TxtinoutReader(tio_dir=sim_dir)

# --- Step-wise Configurations ---

#1. Update timeline in `time.sim` file
sim_reader.set_simulation_period(
    begin_date='01-Jan-2015',        # control simulation period here
    end_date='30-Apr-2026'
)

#2. Set warm-up years in `print.prt` file
sim_reader.set_warmup_year(                 # control warmup period here
    warmup=2                          
)

#3. Ensure simulation outputs for `channel_sd` object in `print.prt` file  
sim_reader.enable_object_in_print_prt(
    obj='channel_sd',
    daily=True,
    monthly=True,
    yearly=True,
    avann=True,
)

# Set print interval in `print.prt` file
sim_reader.set_print_interval(
    interval=1
)

# Set print period in `print.prt` file  
sim_reader.set_print_period(
    begin_date='01-Jan-2015',               # you can control print period here
    end_date='30-Apr-2026'
)

# --- Parameters (Commented out for now) ---
# parameters = [
#     {
#         'name': 'esco',
#         'change_type': 'absval',
#         'value': 0.5
#     }
# ]

# --- Execute SWAT+ Simulation ---
print(f"Starting SWAT+ Simulation in directory: {os.path.basename(sim_dir)}")

# Start the timer
start_time = time.time()

sim_reader.run_swat()

# Stop the timer
end_time = time.time()

# Calculate execution duration
execution_duration = end_time - start_time
minutes, seconds = divmod(execution_duration, 60)

print("Simulation Completed successfully.")
print(f"Total Simulation Execution Time: {int(minutes)} minutes and {seconds:.2f} seconds.")

In [ ]:
# ==========================================
# CELL 5: pySWATPlus AUTOMATED RUN
# ==========================================
import pySWATPlus
import numpy
import pandas
import os
import datetime
import time  # Added to track execution time

# --- SWAT+ Simulation Setup (Automated Directories) ---
base_dir = "/home/jovyan/Taki_Thesis/summery"
today_str = datetime.datetime.now().strftime('%Y-%m-%d')
time_str = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

# Dynamically initialize the original TxtInOut directory path (from Cell 4)
txtinout_dir = os.path.join(base_dir, f"TxtInOut_Full_Execution_{today_str}")

# Initialize TxtinoutReader class for the source directory
txtinout_reader = pySWATPlus.TxtinoutReader(tio_dir=txtinout_dir)

# --- Simulation in a Custom Directory (Automated timestamp) ---
sim_dir = os.path.join(base_dir, f"prediction_model_run_pyswatplus_{time_str}")

if not os.path.exists(sim_dir):
    os.makedirs(sim_dir)

# Copy required input files and executable to the custom directory
txtinout_reader.copy_required_files(sim_dir=sim_dir)

# Initialize TxtinoutReader for the custom directory setup
sim_reader = pySWATPlus.TxtinoutReader(tio_dir=sim_dir)

# --- Determine Automated End Date ---
if 'df_forecast' in globals() and not df_forecast.empty:
    # Use the max date from forecast, minus 1 day to ensure full data coverage
    safe_end = df_forecast.index.max() - datetime.timedelta(days=1)
    auto_end_date = safe_end.strftime('%d-%b-%Y')
else:
    # Fallback date if forecast data is missing
    auto_end_date = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%d-%b-%Y')

print(f"Automated Simulation End Date set to: {auto_end_date}")

# --- Step-wise Configurations ---

#1. Update timeline in `time.sim` file
sim_reader.set_simulation_period(
    begin_date='01-Jan-2015',        # control simulation period here
    end_date=auto_end_date
)

#2. Set warm-up years in `print.prt` file
sim_reader.set_warmup_year(                 # control warmup period here
    warmup=2                          
)

#3. Ensure simulation outputs for `channel_sd` object in `print.prt` file  
sim_reader.enable_object_in_print_prt(
    obj='channel_sd',
    daily=True,
    monthly=True,
    yearly=True,
    avann=True,
)

# Set print interval in `print.prt` file
sim_reader.set_print_interval(
    interval=1
)

# Set print period in `print.prt` file  
sim_reader.set_print_period(
    begin_date='01-Jan-2015',               # you can control print period here
    end_date=auto_end_date
)

# --- Parameters (Commented out for now) ---
# parameters = [
#     {
#         'name': 'esco',
#         'change_type': 'absval',
#         'value': 0.5
#     }
# ]

# --- Execute SWAT+ Simulation ---
print(f"Starting SWAT+ Simulation in directory: {os.path.basename(sim_dir)}")

# Start the timer
start_time = time.time()

sim_reader.run_swat()

# Stop the timer
end_time = time.time()

# Calculate execution duration
execution_duration = end_time - start_time
minutes, seconds = divmod(execution_duration, 60)

print("Simulation Completed successfully.")
print(f"Total Simulation Execution Time: {int(minutes)} minutes and {seconds:.2f} seconds.")

## reading prediction output

In [ ]:
# ==========================================
# CELL 6: CALCULATE AND EXTRACT TN, TP
# ==========================================
import pandas as pd
import os

# Define the path to the output file generated by pySWATPlus
output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")  # Absolute path of simulation directory can be used alternatevely
target_unit = 891

print(f"Reading output from: {output_file_path}")

if os.path.exists(output_file_path):
    try:
        # Read the file
        df_results = pd.read_csv(output_file_path, sep='\s+', skiprows=1)
        
        # Identify the unit column dynamically
        if 'unit' in df_results.columns:
            unit_col = 'unit'
        elif 'gis_id' in df_results.columns:
            unit_col = 'gis_id'
        else:
            unit_col = df_results.columns[4]
            
        # Filter the data for the specific unit
        df_filtered = df_results[df_results[unit_col] == target_unit].copy()
            
        if not df_filtered.empty:
            # Extract the last 3 rows (last 3 Julian days)
            last_3_days = df_filtered.tail(3).copy()
            
            # Define SWAT+ nutrient components
            n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
            p_components = ['orgp_out', 'solp_out']
            
            # Ensure we only use columns that actually exist in your SWAT+ version
            valid_n_cols = [col for col in n_components if col in last_3_days.columns]
            valid_p_cols = [col for col in p_components if col in last_3_days.columns]
            
            # Calculate Total N and Total P by summing the component columns
            last_3_days['Calculated_TN'] = last_3_days[valid_n_cols].sum(axis=1)
            last_3_days['Calculated_TP'] = last_3_days[valid_p_cols].sum(axis=1)
            
            # Build the list of columns to display
            base_cols = [col for col in ['yr', 'mon', 'day', 'jday', 'flo_out'] if col in last_3_days.columns]
            display_cols = base_cols + valid_n_cols + ['Calculated_TN'] + valid_p_cols + ['Calculated_TP']
            
            # Display the focused table
            print(f"\n--- Output for Unit {target_unit} with Calculated TN and TP ---")
            display(last_3_days[display_cols])
            
        else:
            print(f"Notice: No data found for unit {target_unit}.")
            
    except Exception as e:
        print(f"Error reading the file: {e}")
else:
    print(f"Error: Output file not found at {output_file_path}")

In [ ]:
download all data

In [ ]:
# ==========================================
# CELL 6: EXPORT ALL DATA FOR UNIT 891 TO CSV
# ==========================================
import pandas as pd
import os

# Define the path to the output file generated by pySWATPlus
output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")
target_unit = 891

# Define the output CSV filename
csv_output_path = os.path.join(sim_dir, f"unit_{target_unit}_all_data_export.csv")

print(f"Reading output from: {output_file_path}")

if os.path.exists(output_file_path):
    try:
        # Read the raw SWAT+ output file
        df_results = pd.read_csv(output_file_path, sep='\s+', skiprows=1)
        
        # Identify the unit column dynamically
        if 'unit' in df_results.columns:
            unit_col = 'unit'
        elif 'gis_id' in df_results.columns:
            unit_col = 'gis_id'
        else:
            unit_col = df_results.columns[4]
            
        # Filter all data for the specific unit (891)
        df_filtered = df_results[df_results[unit_col] == target_unit].copy()
            
        if not df_filtered.empty:
            # Define nutrient components for calculation
            n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
            p_components = ['orgp_out', 'solp_out']
            
            # Verify existing columns in this specific SWAT+ run
            valid_n_cols = [col for col in n_components if col in df_filtered.columns]
            valid_p_cols = [col for col in p_components if col in df_filtered.columns]
            
            # Calculate Total N and Total P for all rows
            df_filtered['Calculated_TN'] = df_filtered[valid_n_cols].sum(axis=1)
            df_filtered['Calculated_TP'] = df_filtered[valid_p_cols].sum(axis=1)
            
            # Define the final column list for the CSV export
            base_cols = [col for col in ['yr', 'mon', 'day', 'jday', 'flo_out'] if col in df_filtered.columns]
            export_cols = base_cols + valid_n_cols + ['Calculated_TN'] + valid_p_cols + ['Calculated_TP']
            
            # Select relevant columns and save to CSV
            df_final_export = df_filtered[export_cols]
            df_final_export.to_csv(csv_output_path, index=False)
            
            print(f"Success: All data for unit {target_unit} has been exported.")
            print(f"CSV File Path: {csv_output_path}")
            
            # Display the first 5 rows of the exported data as a preview
            print("\n--- Preview of exported data (First 5 rows) ---")
            display(df_final_export.head(5))
            
        else:
            print(f"Notice: No data found for unit {target_unit} to export.")
            
    except Exception as e:
        print(f"Error processing the file: {e}")
else:
    print(f"Error: Output file not found at {output_file_path}")

In [ ]:
# ==========================================
# CELL 7: TIME SERIES PLOTTING (FLOW, N, P)
# ==========================================
import matplotlib.pyplot as plt
import pandas as pd
import os

# Ensure df_final_export exists in memory
if 'df_final_export' in locals():
    df_plot = df_final_export.copy()
    
    # Map the existing column names to what pandas expects (year, month, day)
    # a dictionary is created to help pandas assemble the date
    date_columns = {
        'year': df_plot['yr'],
        'month': df_plot['mon'],
        'day': df_plot['day']
    }
    
    # Create the Date column correctly
    df_plot['Date'] = pd.to_datetime(date_columns)
    df_plot = df_plot.sort_values('Date')

    # --- 1. Flow Out Plot ---
    plt.figure(figsize=(10, 5))
    plt.plot(df_plot['Date'], df_plot['flo_out'], color='blue', label='Flow Out')
    plt.xlabel('Date')
    plt.ylabel('Flow ($m^3/s$)')
    plt.title('Time Series of Flow Out (Unit 891)')
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_flow_out.png')
    plt.close()
    print("Generated: plot_flow_out.png")

    # --- 2. Nitrogen Components Plot ---
    plt.figure(figsize=(12, 6))
    plt.plot(df_plot['Date'], df_plot['Calculated_TN'], color='black', linewidth=2, label='Total N (Calculated)')
    
    # Plot components if they exist
    n_comp_list = [('orgn_out', 'Organic N', '--'), ('no3_out', 'NO3 Out', ':'), 
                   ('no2_out', 'NO2 Out', '-.'), ('nh3_out', 'NH3 Out', '-')]
    
    for col, label, style in n_comp_list:
        if col in df_plot.columns:
            plt.plot(df_plot['Date'], df_plot['orgn_out'], label=label, linestyle=style, alpha=0.7)
    
    plt.xlabel('Date')
    plt.ylabel('Nitrogen (kg)')
    plt.title('Nitrogen Components Time Series (Unit 891)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_nitrogen_components.png')
    plt.close()
    print("Generated: plot_nitrogen_components.png")

    # --- 3. Phosphorus Components Plot ---
    plt.figure(figsize=(12, 6))
    plt.plot(df_plot['Date'], df_plot['Calculated_TP'], color='red', linewidth=2, label='Total P (Calculated)')
    
    if 'orgp_out' in df_plot.columns: 
        plt.plot(df_plot['Date'], df_plot['orgp_out'], label='Organic P', linestyle='--', alpha=0.7)
    if 'solp_out' in df_plot.columns: 
        plt.plot(df_plot['Date'], df_plot['solp_out'], label='Soluble P', linestyle=':', alpha=0.7)
    
    plt.xlabel('Date')
    plt.ylabel('Phosphorus (kg)')
    plt.title('Phosphorus Components Time Series (Unit 891)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_phosphorus_components.png')
    plt.close()
    print("Generated: plot_phosphorus_components.png")

else:
    print("Error: df_final_export not found. Please run Cell 6 first.")

In [ ]:
!pip install folium plotly ipywidgets

In [ ]:
# ==========================================
# POST-PROCESSING & INTERACTIVE DASHBOARD (PLOTLY NATIVE)
# ==========================================
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import folium
from IPython.display import display

# --- 1. Dynamic Data Loading & Processing ---
try:
    sim_dir
except NameError:
    base_dir = "/home/jovyan/Taki_Thesis/summery"
    list_of_dirs = glob.glob(os.path.join(base_dir, "prediction_model_run_pyswatplus_*"))
    if list_of_dirs:
        sim_dir = max(list_of_dirs, key=os.path.getctime)
        print(f"Auto-detected latest simulation directory: {os.path.basename(sim_dir)}")
    else:
        raise FileNotFoundError("No simulation directories found. Please run the model first.")

output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")
target_unit = 891

try:
    # --- EXACT MATCH LOGIC FROM YOUR TABULAR CODE ---
    df_results = pd.read_csv(output_file_path, sep=r'\s+', skiprows=1)
    
    if 'unit' in df_results.columns:
        unit_col = 'unit'
    elif 'gis_id' in df_results.columns:
        unit_col = 'gis_id'
    else:
        unit_col = df_results.columns[4]
        
    df_filtered = df_results[df_results[unit_col] == target_unit].copy()
    
    # Standardize Dates
    df_filtered['Date'] = pd.to_datetime(df_filtered[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'}))
    
    # Calculate TN and TP matching your exact components
    n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
    p_components = ['orgp_out', 'solp_out']  # Fixed: changed from sedp_out to orgp_out
    
    valid_n_cols = [col for col in n_components if col in df_filtered.columns]
    valid_p_cols = [col for col in p_components if col in df_filtered.columns]
    
    df_filtered['TN'] = df_filtered[valid_n_cols].sum(axis=1)
    df_filtered['TP'] = df_filtered[valid_p_cols].sum(axis=1)
    
    # Extract the last 3 rows (last 3 Julian days)
    df_forecast = df_filtered.tail(3).copy()
    print("Simulation data successfully loaded and matched with tabular data!")

except Exception as e:
    print(f"Error loading data: {e}. Using fallback data for layout testing.")
    df_forecast = pd.DataFrame({
        'Date': pd.date_range(start='2026-04-28', periods=3),
        'flo_out': [0, 0, 0], 'TN': [0, 0, 0], 'TP': [0, 0, 0]
    })

# --- 2. Build Interactive Plotly Chart with Native Dropdown ---
fig = go.Figure()

# Add traces for Flow, TN, and TP
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['flo_out'], mode='lines+markers', name='Flow', visible=True, line=dict(color='firebrick', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TN'], mode='lines+markers', name='TN', visible=False, line=dict(color='teal', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TP'], mode='lines+markers', name='TP', visible=False, line=dict(color='purple', width=4)))

# Add the dropdown menu to toggle visibility
fig.update_layout(
    title="3-Day Forecast: Flow (m³/s)",
    xaxis_title="Date",
    yaxis_title="Value",
    template='plotly_white',
    updatemenus=[dict(
        active=0,
        buttons=list([
            dict(label="Flow (m³/s)",
                 method="update",
                 args=[{"visible": [True, False, False]}, {"title": "3-Day Forecast: Flow (m³/s)"}]),
            dict(label="Total Nitrogen (kg)",
                 method="update",
                 args=[{"visible": [False, True, False]}, {"title": "3-Day Forecast: Total Nitrogen (kg)"}]),
            dict(label="Total Phosphorus (kg)",
                 method="update",
                 args=[{"visible": [False, False, True]}, {"title": "3-Day Forecast: Total Phosphorus (kg)"}]),
        ]),
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0,
        xanchor="left",
        y=1.15,
        yanchor="top"
    )]
)

fig.show()

# --- 3. Display the Static Map below the chart ---
m = folium.Map(location=[66.36, 29.32], zoom_start=11, tiles='cartodbpositron')
folium.Marker([66.36, 29.32], popup="Oulanka Catchment Outlet (891)", icon=folium.Icon(color='blue', icon='info-sign')).add_to(m)
display(m)

In [ ]:
# ==========================================
# FULL DASHBOARD: LARGE HYDROGRAPH + ENHANCED MAP
# ==========================================
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import folium
from folium import plugins
import json
from IPython.display import display

# --- 1. Dynamic Data Loading & Processing ---
try:
    sim_dir
except NameError:
    base_dir = "/home/jovyan/Taki_Thesis/summery"
    list_of_dirs = glob.glob(os.path.join(base_dir, "prediction_model_run_pyswatplus_*"))
    if list_of_dirs:
        sim_dir = max(list_of_dirs, key=os.path.getctime)
        print(f"Auto-detected simulation directory: {os.path.basename(sim_dir)}")
    else:
        raise FileNotFoundError("No simulation directories found. Please run the model first.")

output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")
target_unit = 891

try:
    df_results = pd.read_csv(output_file_path, sep=r'\s+', skiprows=1)
    
    if 'unit' in df_results.columns:
        unit_col = 'unit'
    elif 'gis_id' in df_results.columns:
        unit_col = 'gis_id'
    else:
        unit_col = df_results.columns[4]
        
    df_filtered = df_results[df_results[unit_col] == target_unit].copy()
    
    # Standardize Dates
    df_filtered['Date'] = pd.to_datetime(df_filtered[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'}))
    
    # Calculate TN and TP matching your exact components
    n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
    p_components = ['orgp_out', 'solp_out'] 
    
    valid_n_cols = [col for col in n_components if col in df_filtered.columns]
    valid_p_cols = [col for col in p_components if col in df_filtered.columns]
    
    df_filtered['TN'] = df_filtered[valid_n_cols].sum(axis=1)
    df_filtered['TP'] = df_filtered[valid_p_cols].sum(axis=1)
    
    df_forecast = df_filtered.tail(3).copy()
    print("Dashboard Data successfully loaded!")

except Exception as e:
    print(f"Error loading data: {e}. Using fallback data.")
    df_forecast = pd.DataFrame({
        'Date': pd.date_range(start='2026-04-28', periods=3),
        'flo_out': [0, 0, 0], 'TN': [0, 0, 0], 'TP': [0, 0, 0]
    })

# --- 2. Build Large Interactive Plotly Chart with Dropdown ---
fig = go.Figure()

# Add traces for Flow, TN, and TP
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['flo_out'], mode='lines+markers', name='Flow', visible=True, line=dict(color='firebrick', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TN'], mode='lines+markers', name='TN', visible=False, line=dict(color='teal', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TP'], mode='lines+markers', name='TP', visible=False, line=dict(color='purple', width=4)))

# Add the dropdown menu to toggle visibility
fig.update_layout(
    title="3-Day Forecast Dashboard",
    xaxis_title="Date",
    yaxis_title="Value",
    template='plotly_white',
    updatemenus=[dict(
        active=0,
        buttons=list([
            dict(label="Flow (m³/s)", method="update", args=[{"visible": [True, False, False]}, {"title": "3-Day Forecast: Flow (m³/s)"}]),
            dict(label="Total Nitrogen (kg)", method="update", args=[{"visible": [False, True, False]}, {"title": "3-Day Forecast: Total Nitrogen (kg)"}]),
            dict(label="Total Phosphorus (kg)", method="update", args=[{"visible": [False, False, True]}, {"title": "3-Day Forecast: Total Phosphorus (kg)"}]),
        ]),
        direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0.0, xanchor="left", y=1.15, yanchor="top"
    )]
)

fig.show()

# --- 3. Display the Enhanced Interactive Map with Embedded Hydrograph ---
table_rows = ""
for index, row in df_forecast.iterrows():
    date_str = row['Date'].strftime('%b %d')
    flow = round(row['flo_out'], 2)
    tn = round(row['TN'], 2)
    tp = round(row['TP'], 2)
    
    table_rows += f"""
    <tr>
        <td style="padding: 4px; border-bottom: 1px solid #ddd; text-align: left;"><b>{date_str}</b></td>
        <td style="padding: 4px; border-bottom: 1px solid #ddd; color: #C0392B;">{flow}</td>
        <td style="padding: 4px; border-bottom: 1px solid #ddd; color: #16A085;">{tn}</td>
        <td style="padding: 4px; border-bottom: 1px solid #ddd; color: #8E44AD;">{tp}</td>
    </tr>
    """

dates_js = json.dumps(df_forecast['Date'].dt.strftime('%b %d').tolist())
flow_js = json.dumps(df_forecast['flo_out'].tolist())

iframe_html = f"""
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
</head>
<body style="font-family: Arial; margin: 0px; padding: 5px; background-color: white;">
    <h4 style="margin-top:0px; margin-bottom:8px; color: #2C3E50; text-align: center;"><b>Outlet 891 Forecast</b></h4>
    <table style="width: 100%; border-collapse: collapse; font-size: 12px; text-align: center;">
        <tr style="background-color: #ECF0F1;">
            <th style="padding: 4px; border-bottom: 2px solid #BDC3C7; text-align: left;">Date</th>
            <th style="padding: 4px; border-bottom: 2px solid #BDC3C7;">Flow</th>
            <th style="padding: 4px; border-bottom: 2px solid #BDC3C7;">TN</th>
            <th style="padding: 4px; border-bottom: 2px solid #BDC3C7;">TP</th>
        </tr>
        {table_rows}
    </table>
    <div id="mini-hydrograph" style="width: 100%; height: 180px; margin-top: 15px;"></div>
    <script>
        var trace = {{
            x: {dates_js}, y: {flow_js}, type: 'scatter', mode: 'lines+markers',
            line: {{color: 'firebrick', width: 3}}, marker: {{size: 8}}
        }};
        var layout = {{
            title: {{text: '3-Day Flow (m³/s)', font: {{size: 13, color: '#34495E'}}}},
            margin: {{l: 30, r: 10, t: 25, b: 20}}, paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: 'rgba(0,0,0,0)',
            xaxis: {{showgrid: false, tickfont: {{size: 10}}}}, yaxis: {{showgrid: true, gridcolor: '#ecf0f1', tickfont: {{size: 10}}}}
        }};
        Plotly.newPlot('mini-hydrograph', [trace], layout, {{displayModeBar: false}});
    </script>
</body>
</html>
"""

iframe = folium.IFrame(html=iframe_html, width=380, height=360)
smart_popup = folium.Popup(iframe, max_width=380)

m = folium.Map(location=[66.36, 29.32], zoom_start=11, tiles=None)

folium.TileLayer('cartodbpositron', name='Light Map').add_to(m)
folium.TileLayer('OpenStreetMap', name='Street Map').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite View', overlay=False
).add_to(m)

folium.Marker(
    [66.36, 29.32], popup=smart_popup, icon=folium.Icon(color='red', icon='info-sign'), tooltip="Click to view Data Table & Hydrograph!"
).add_to(m)

folium.Circle(
    radius=3000, location=[66.36, 29.32], popup='Immediate Catchment Outlet Zone',
    color='#3186cc', fill=True, fill_color='#3186cc', fill_opacity=0.2
).add_to(m)

folium.LayerControl(position='topright').add_to(m)
plugins.Fullscreen(position='topleft').add_to(m)

display(m)

In [ ]:
# ==========================================
# ULTIMATE DASHBOARD: HYDROGRAPH + GIS MAP (WITH SHAPEFILES)
# ==========================================
import os
import glob
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import folium
from folium import plugins
import json
from IPython.display import display

# --- 1. Dynamic Data Loading & Processing ---
try:
    sim_dir
except NameError:
    base_dir = "/home/jovyan/Taki_Thesis/summery"
    list_of_dirs = glob.glob(os.path.join(base_dir, "prediction_model_run_pyswatplus_*"))
    if list_of_dirs:
        sim_dir = max(list_of_dirs, key=os.path.getctime)
    else:
        raise FileNotFoundError("No simulation directories found. Please run the model first.")

output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")
target_unit = 891

try:
    df_results = pd.read_csv(output_file_path, sep=r'\s+', skiprows=1)
    unit_col = 'unit' if 'unit' in df_results.columns else ('gis_id' if 'gis_id' in df_results.columns else df_results.columns[4])
    df_filtered = df_results[df_results[unit_col] == target_unit].copy()
    
    df_filtered['Date'] = pd.to_datetime(df_filtered[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'}))
    
    n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
    p_components = ['orgp_out', 'solp_out'] 
    
    valid_n_cols = [col for col in n_components if col in df_filtered.columns]
    valid_p_cols = [col for col in p_components if col in df_filtered.columns]
    
    df_filtered['TN'] = df_filtered[valid_n_cols].sum(axis=1)
    df_filtered['TP'] = df_filtered[valid_p_cols].sum(axis=1)
    
    df_forecast = df_filtered.tail(3).copy()
    print("Hydrological Data loaded successfully!")
except Exception as e:
    print(f"Error loading time-series data: {e}")
    df_forecast = pd.DataFrame({'Date': pd.date_range(start='2026-04-28', periods=3), 'flo_out': [0,0,0], 'TN': [0,0,0], 'TP': [0,0,0]})

# --- 2. Build Large Interactive Plotly Chart ---
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['flo_out'], mode='lines+markers', name='Flow', visible=True, line=dict(color='firebrick', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TN'], mode='lines+markers', name='TN', visible=False, line=dict(color='teal', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TP'], mode='lines+markers', name='TP', visible=False, line=dict(color='purple', width=4)))

fig.update_layout(
    title="3-Day Forecast Dashboard", xaxis_title="Date", yaxis_title="Value", template='plotly_white',
    updatemenus=[dict(
        active=0,
        buttons=list([
            dict(label="Flow (m³/s)", method="update", args=[{"visible": [True, False, False]}, {"title": "3-Day Forecast: Flow (m³/s)"}]),
            dict(label="Total Nitrogen (kg)", method="update", args=[{"visible": [False, True, False]}, {"title": "3-Day Forecast: Total Nitrogen (kg)"}]),
            dict(label="Total Phosphorus (kg)", method="update", args=[{"visible": [False, False, True]}, {"title": "3-Day Forecast: Total Phosphorus (kg)"}]),
        ]),
        direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0.0, xanchor="left", y=1.15, yanchor="top"
    )]
)
fig.show()

# --- 3. Build GIS Map with Shapefile Layers ---

# Initialize Map
m = folium.Map(location=[66.36, 29.32], zoom_start=10, tiles=None)

# Add Basemaps (Radio buttons in Layer Control)
folium.TileLayer('cartodbpositron', name='Light Basemap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite Imagery', overlay=False
).add_to(m)

# Load and Add Shapefiles as Overlays (Checkboxes in Layer Control)
shape_dir = "/home/jovyan/Taki_Thesis/latest_model/Watershed/Shapes"

try:
    # 3a. Watershed Boundary
    wshed_path = os.path.join(shape_dir, "dem_fwshed_dem.shp")
    if os.path.exists(wshed_path):
        wshed_gdf = gpd.read_file(wshed_path).to_crs(epsg=4326)
        folium.GeoJson(
            wshed_gdf, name="Catchment Boundary",
            style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}
        ).add_to(m)

    # 3b. Subbasins
    subs_path = os.path.join(shape_dir, "subs1.shp")
    if os.path.exists(subs_path):
        subs_gdf = gpd.read_file(subs_path).to_crs(epsg=4326)
        folium.GeoJson(
            subs_gdf, name="Subbasins",
            style_function=lambda x: {'color': 'brown', 'weight': 1, 'fillColor': 'yellow', 'fillOpacity': 0.1}
        ).add_to(m)

    # 3c. River Network
    rivs_path = os.path.join(shape_dir, "rivs1.shp")
    if os.path.exists(rivs_path):
        rivs_gdf = gpd.read_file(rivs_path).to_crs(epsg=4326)
        folium.GeoJson(
            rivs_gdf, name="River Network",
            style_function=lambda x: {'color': 'blue', 'weight': 2}
        ).add_to(m)
        
    print("GIS Shapefiles loaded successfully!")
except Exception as e:
    print(f"Warning: Could not load some shapefiles. Error: {e}")

# --- 4. Add the Smart Popup & Marker ---
table_rows = ""
for index, row in df_forecast.iterrows():
    date_str = row['Date'].strftime('%b %d')
    table_rows += f"<tr><td style='padding: 2px;'>{date_str}</td><td style='color:#C0392B;'>{round(row['flo_out'],2)}</td><td style='color:#16A085;'>{round(row['TN'],2)}</td><td style='color:#8E44AD;'>{round(row['TP'],2)}</td></tr>"

dates_js, flow_js = json.dumps(df_forecast['Date'].dt.strftime('%b %d').tolist()), json.dumps(df_forecast['flo_out'].tolist())

iframe_html = f"""
<!DOCTYPE html><html><head><script src="https://cdn.plot.ly/plotly-latest.min.js"></script></head>
<body style="font-family: Arial; margin: 0; padding: 5px;">
    <h4 style="text-align: center; margin: 0 0 5px 0;">Outlet 891 Forecast</h4>
    <table style="width: 100%; border-collapse: collapse; font-size: 11px; text-align: center;">
        <tr style="background-color: #ECF0F1;"><th>Date</th><th>Flow</th><th>TN</th><th>TP</th></tr>{table_rows}
    </table>
    <div id="mini-plot" style="width: 100%; height: 160px; margin-top: 5px;"></div>
    <script>
        Plotly.newPlot('mini-plot', [{{x: {dates_js}, y: {flow_js}, type: 'scatter', mode: 'lines+markers', line: {{color: 'firebrick'}}}}], 
        {{margin: {{l: 25, r: 10, t: 10, b: 20}}}}, {{displayModeBar: false}});
    </script>
</body></html>
"""

folium.Marker(
    [66.36, 29.32], 
    popup=folium.Popup(folium.IFrame(html=iframe_html, width=320, height=300), max_width=320), 
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

# Add Plugins (This adds the Dropdown/Layer Control)
folium.LayerControl(position='topright', collapsed=False).add_to(m)
plugins.Fullscreen(position='topleft').add_to(m)

display(m)

## 

In [ ]:
# ==========================================
# ULTIMATE DASHBOARD: HYDROGRAPH + GIS MAP (COLORBLIND FRIENDLY & PROFESSIONAL)
# ==========================================
import os
import glob
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import folium
from folium import plugins
import json
from IPython.display import display

# --- 1. Dynamic Data Loading & Processing ---
try:
    sim_dir
except NameError:
    base_dir = "/home/jovyan/Taki_Thesis/summery"
    list_of_dirs = glob.glob(os.path.join(base_dir, "prediction_model_run_pyswatplus_*"))
    if list_of_dirs:
        sim_dir = max(list_of_dirs, key=os.path.getctime)
    else:
        raise FileNotFoundError("No simulation directories found. Please run the model first.")

output_file_path = os.path.join(sim_dir, "channel_sd_day.txt")
target_unit = 891

try:
    df_results = pd.read_csv(output_file_path, sep=r'\s+', skiprows=1)
    unit_col = 'unit' if 'unit' in df_results.columns else ('gis_id' if 'gis_id' in df_results.columns else df_results.columns[4])
    df_filtered = df_results[df_results[unit_col] == target_unit].copy()
    
    df_filtered['Date'] = pd.to_datetime(df_filtered[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'}))
    
    n_components = ['orgn_out', 'no3_out', 'no2_out', 'nh3_out']
    p_components = ['orgp_out', 'solp_out'] 
    
    valid_n_cols = [col for col in n_components if col in df_filtered.columns]
    valid_p_cols = [col for col in p_components if col in df_filtered.columns]
    
    df_filtered['TN'] = df_filtered[valid_n_cols].sum(axis=1)
    df_filtered['TP'] = df_filtered[valid_p_cols].sum(axis=1)
    
    df_forecast = df_filtered.tail(3).copy()
    print("Hydrological Data loaded successfully!")
except Exception as e:
    print(f"Error loading time-series data: {e}")
    df_forecast = pd.DataFrame({'Date': pd.date_range(start='2026-04-28', periods=3), 'flo_out': [0,0,0], 'TN': [0,0,0], 'TP': [0,0,0]})

# --- 2. Build Large Interactive Plotly Chart ---
fig = go.Figure()

# Using Okabe-Ito colorblind-friendly palette
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['flo_out'], mode='lines+markers', name='Flow', visible=True, line=dict(color='#0072B2', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TN'], mode='lines+markers', name='TN', visible=False, line=dict(color='#009E73', width=4)))
fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast['TP'], mode='lines+markers', name='TP', visible=False, line=dict(color='#D55E00', width=4)))

fig.update_layout(
    title="3-Day Forecast Dashboard", xaxis_title="Date", yaxis_title="Value", template='plotly_white',
    updatemenus=[dict(
        active=0,
        buttons=list([
            dict(label="Flow (m³/s)", method="update", args=[{"visible": [True, False, False]}, {"title": "3-Day Forecast: Flow (m³/s)"}]),
            dict(label="Total Nitrogen (kg)", method="update", args=[{"visible": [False, True, False]}, {"title": "3-Day Forecast: Total Nitrogen (kg)"}]),
            dict(label="Total Phosphorus (kg)", method="update", args=[{"visible": [False, False, True]}, {"title": "3-Day Forecast: Total Phosphorus (kg)"}]),
        ]),
        direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0.0, xanchor="left", y=1.15, yanchor="top"
    )]
)
fig.show()

# --- 3. Build GIS Map with Shapefile Layers ---

# Initialize Map - starting with CartoDB Positron for a cleaner, professional look
m = folium.Map(location=[66.36, 29.32], zoom_start=10, tiles='cartodbpositron')

# Add alternative Basemaps
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite Imagery', overlay=False
).add_to(m)

# Load and Add Shapefiles as Overlays
shape_dir = "/home/jovyan/Taki_Thesis/latest_model/Watershed/Shapes"

try:
    # 3a. Watershed Boundary (Thick deep blue border, no fill)
    wshed_path = os.path.join(shape_dir, "dem_fwshed_dem.shp")
    if os.path.exists(wshed_path):
        wshed_gdf = gpd.read_file(wshed_path).to_crs(epsg=4326)
        folium.GeoJson(
            wshed_gdf, name="Catchment Boundary",
            style_function=lambda x: {'color': '#004C99', 'weight': 3, 'fillOpacity': 0}
        ).add_to(m)

    # 3b. Subbasins (Thin dark grey border, very subtle light fill)
    subs_path = os.path.join(shape_dir, "subs1.shp")
    if os.path.exists(subs_path):
        subs_gdf = gpd.read_file(subs_path).to_crs(epsg=4326)
        folium.GeoJson(
            subs_gdf, name="Subbasins",
            style_function=lambda x: {'color': '#555555', 'weight': 0.8, 'fillColor': '#E6E6E6', 'fillOpacity': 0.15}
        ).add_to(m)

    # 3c. River Network (Clear blue lines)
    rivs_path = os.path.join(shape_dir, "rivs1.shp")
    if os.path.exists(rivs_path):
        rivs_gdf = gpd.read_file(rivs_path).to_crs(epsg=4326)
        folium.GeoJson(
            rivs_gdf, name="River Network",
            style_function=lambda x: {'color': '#0072B2', 'weight': 1.5}
        ).add_to(m)
        
    print("GIS Shapefiles loaded successfully with professional styling!")
except Exception as e:
    print(f"Warning: Could not load some shapefiles. Error: {e}")

# --- 4. Add the Smart Popup & Marker ---
table_rows = ""
for index, row in df_forecast.iterrows():
    date_str = row['Date'].strftime('%b %d')
    table_rows += f"<tr><td style='padding: 2px;'>{date_str}</td><td style='color:#0072B2;'><b>{round(row['flo_out'],2)}</b></td><td style='color:#009E73;'>{round(row['TN'],2)}</td><td style='color:#D55E00;'>{round(row['TP'],2)}</td></tr>"

dates_js, flow_js = json.dumps(df_forecast['Date'].dt.strftime('%b %d').tolist()), json.dumps(df_forecast['flo_out'].tolist())

iframe_html = f"""
<!DOCTYPE html><html><head><script src="https://cdn.plot.ly/plotly-latest.min.js"></script></head>
<body style="font-family: Arial; margin: 0; padding: 5px;">
    <h4 style="text-align: center; margin: 0 0 5px 0; color: #333333;">Outlet 891 Forecast</h4>
    <table style="width: 100%; border-collapse: collapse; font-size: 11px; text-align: center;">
        <tr style="background-color: #F0F0F0; border-bottom: 2px solid #CCCCCC;">
            <th>Date</th><th>Flow</th><th>TN</th><th>TP</th>
        </tr>{table_rows}
    </table>
    <div id="mini-plot" style="width: 100%; height: 160px; margin-top: 5px;"></div>
    <script>
        Plotly.newPlot('mini-plot', [{{x: {dates_js}, y: {flow_js}, type: 'scatter', mode: 'lines+markers', line: {{color: '#0072B2'}}}}], 
        {{margin: {{l: 25, r: 10, t: 10, b: 20}}}}, {{displayModeBar: false}});
    </script>
</body></html>
"""

# Using a distinct red marker for clear visibility
folium.Marker(
    [66.36, 29.32], 
    popup=folium.Popup(folium.IFrame(html=iframe_html, width=320, height=300), max_width=320), 
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

# Add Plugins
folium.LayerControl(position='topright', collapsed=False).add_to(m)
plugins.Fullscreen(position='topleft').add_to(m)

# Finally display the map
display(m)